# Notebook 00 — Quickstart Runner (VS Code + Colab)

Single notebook to execute the full replication pipeline. Runs from
`mini_project/notebooks/00_quickstart_runner.ipynb` and assumes the
`mini_project/` directory is already on disk (no `git clone` needed).

**Pipeline:**
1. Environment check (GPU, PyTorch)
2. Generate Navier-Stokes dataset
3. Architecture comparison (FNO vs F-FNO at depths 4/8/12)
4. Training-trick ablation
5. Generate all plots and summary CSVs

Each stage is gated by a boolean flag so you can re-run individual stages.
All heavy computation writes results to `mini_project/results/` and model
checkpoints to `mini_project/runs/` — so if a stage completes, rerunning
later stages won't re-train.

## Stage 0 — Setup and environment check

In [ ]:
import sys
import time
from pathlib import Path

# Resolve project root. This notebook lives in mini_project/notebooks/,
# so the project root is one level up.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'Expected contents: models/, data/, train.py, notebooks/')
print(f'Actual contents:   {sorted(p.name for p in PROJECT_ROOT.iterdir() if not p.name.startswith("."))}')

In [ ]:
import torch

print(f'Python:   {sys.version.split()[0]}')
print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device:   {torch.cuda.get_device_name(0)}')
    print(f'Memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('!! No GPU detected — data generation will be slow and training will be very slow.')

In [ ]:
# Install missing dependencies. On Colab these are usually already present.
try:
    import matplotlib, numpy, pandas
    print('Core dependencies already installed.')
except ImportError:
    print('Installing dependencies...')
    !pip install --quiet matplotlib numpy pandas

## Control panel

Set which stages to run. The first time through, leave all four True.
For reruns (e.g. re-plotting with already-trained models), flip training
stages to False.

Time estimates on a T4:
- Data generation: ~8 min (one-time cost)
- Depth sweep: ~90 min (6 runs × ~15 min each)
- Ablation: ~90 min (6 runs × ~15 min each)
- Plots: ~30 seconds

In [ ]:
RUN_DATA_GEN    = True
RUN_DEPTH_SWEEP = True
RUN_ABLATION    = True
RUN_PLOTS       = True

# Knobs you might want to tweak if you're short on time / memory:
DEPTHS = [4, 8, 12]           # paper used up to 24, but each layer costs compute
N_EPOCHS_DEPTH = 20           # lower for quicker runs
N_EPOCHS_ABLATION = 15
BATCH_SIZE = 20               # lower to 10 or 8 if you hit GPU OOM
MAX_SECONDS_PER_RUN = 18 * 60 # hard time budget per training run

DATA_DIR = PROJECT_ROOT / 'ns_data'
RUNS_DIR = PROJECT_ROOT / 'runs'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Stage 1 — Data generation

Generates 800 train / 100 val / 100 test Navier-Stokes trajectories at
64×64 resolution, 20 timesteps each. Saves to `mini_project/ns_data/`
as `train.pt`, `val.pt`, `test.pt`, and `metadata.pt`.

Total size on disk: ~100 MB.

In [ ]:
if RUN_DATA_GEN:
    from data.generate_ns import generate_and_save
    t0 = time.time()
    meta = generate_and_save(
        out_dir=str(DATA_DIR),
        n_train=800, n_val=100, n_test=100,
        N=64, T=20,
        dt=1e-3, record_every=100,
        viscosity=1e-3,
        batch_size=50,
        device=device,
        seed=42,
    )
    print(f'\nData generation done in {time.time() - t0:.1f}s')
    print(f'Metadata: {meta}')
else:
    print('Skipping data generation.')
    assert (DATA_DIR / 'train.pt').exists(), 'Data missing! Set RUN_DATA_GEN=True.'

In [ ]:
# Quick visual sanity check of the generated data
import matplotlib.pyplot as plt

train_data = torch.load(DATA_DIR / 'train.pt', weights_only=True)
print(f'Train data shape: {train_data.shape}')
print(f'Vorticity range: [{train_data.min():.3f}, {train_data.max():.3f}]')
print(f'Vorticity std:   {train_data.std():.3f}')

fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
timesteps = [0, 4, 9, 14, 19]
vmax = train_data[0].abs().max().item()
for ax, t in zip(axes, timesteps):
    im = ax.imshow(train_data[0, t], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f't = {t * 0.1:.1f}s')
    ax.axis('off')
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
plt.suptitle('Sample trajectory — vorticity evolution', y=1.02)
plt.savefig(RESULTS_DIR / 'sample_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()
del train_data

## Stage 2 — Architecture comparison

Train FNO and F-FNO at depths `DEPTHS = [4, 8, 12]`. Six total runs.
Results saved to `results/depth_sweep_results.pt`.

In [ ]:
if RUN_DEPTH_SWEEP:
    from train import train, TrainConfig

    all_results = {}
    existing_path = RESULTS_DIR / 'depth_sweep_results.pt'
    if existing_path.exists():
        all_results = torch.load(existing_path, weights_only=False)
        print(f'Loaded {len(all_results)} existing results; will skip those.')

    for model_type in ['fno', 'ffno']:
        for n_layers in DEPTHS:
            key = f'{model_type}_L{n_layers}'
            if key in all_results:
                print(f'[skip] {key} already done (test N-MSE={all_results[key]["test_loss"]:.4f})')
                continue

            print(f'\n{"=" * 60}\nTraining {key}\n{"=" * 60}')
            cfg = TrainConfig(
                model_type=model_type,
                hidden=32, modes=12, n_layers=n_layers,
                data_dir=str(DATA_DIR),
                batch_size=BATCH_SIZE,
                n_epochs=N_EPOCHS_DEPTH,
                lr=2.5e-3, weight_decay=1e-4, warmup_steps=500,
                use_cosine_decay=True,
                input_noise_std=1e-3,
                device=device, seed=0,
                out_dir=str(RUNS_DIR / key),
                max_train_seconds=MAX_SECONDS_PER_RUN,
            )
            all_results[key] = train(cfg)
            torch.save(all_results, existing_path)
    print(f'\nDepth sweep done. {len(all_results)} runs saved to {existing_path}')
else:
    print('Skipping depth sweep.')

## Stage 3 — Training-trick ablation

Fix architecture at 8 layers, toggle cosine LR and input noise independently
for both FNO and F-FNO. Six total runs.
Results saved to `results/ablation_results.pt`.

In [ ]:
if RUN_ABLATION:
    from train import train, TrainConfig

    configs = [
        {'name': 'FNO (plain)',             'model_type': 'fno',  'noise': 0.0,  'cosine': False},
        {'name': 'FNO + cosine LR',         'model_type': 'fno',  'noise': 0.0,  'cosine': True},
        {'name': 'FNO + cosine + noise',    'model_type': 'fno',  'noise': 1e-3, 'cosine': True},
        {'name': 'F-FNO (plain)',           'model_type': 'ffno', 'noise': 0.0,  'cosine': False},
        {'name': 'F-FNO + cosine LR',       'model_type': 'ffno', 'noise': 0.0,  'cosine': True},
        {'name': 'F-FNO + cosine + noise',  'model_type': 'ffno', 'noise': 1e-3, 'cosine': True},
    ]

    ablation_results = {}
    existing_path = RESULTS_DIR / 'ablation_results.pt'
    if existing_path.exists():
        ablation_results = torch.load(existing_path, weights_only=False)
        print(f'Loaded {len(ablation_results)} existing ablation results.')

    for i, cfg_dict in enumerate(configs):
        name = cfg_dict['name']
        if name in ablation_results:
            print(f'[skip] {name} done (test N-MSE={ablation_results[name]["test_loss"]:.4f})')
            continue

        print(f'\n{"=" * 60}\n[{i+1}/6] {name}\n{"=" * 60}')
        cfg = TrainConfig(
            model_type=cfg_dict['model_type'],
            hidden=32, modes=12, n_layers=8,
            data_dir=str(DATA_DIR),
            batch_size=BATCH_SIZE,
            n_epochs=N_EPOCHS_ABLATION,
            lr=2.5e-3, weight_decay=1e-4, warmup_steps=500,
            use_cosine_decay=cfg_dict['cosine'],
            input_noise_std=cfg_dict['noise'],
            device=device, seed=0,
            out_dir=str(RUNS_DIR / 'ablation' / f'run_{i}'),
            max_train_seconds=MAX_SECONDS_PER_RUN,
        )
        ablation_results[name] = train(cfg)
        torch.save(ablation_results, existing_path)
    print(f'\nAblation done. {len(ablation_results)} runs saved.')
else:
    print('Skipping ablation.')

## Stage 4 — Plots and summary tables

Builds every plot and CSV needed for the report: parameter counts,
depth sweep, ablation, training curves, rollout error, and a qualitative
side-by-side comparison.

In [ ]:
if RUN_PLOTS:
    import numpy as np
    import pandas as pd
    from models import FNO2d, FFNO2d

    # ---------- Plot 1: parameter counts ----------
    rows = []
    for n_layers in [4, 8, 12, 16, 24]:
        fno = FNO2d(n_layers=n_layers, hidden=32, modes1=12, modes2=12)
        ffno = FFNO2d(n_layers=n_layers, hidden=32, modes_x=12, modes_y=12)
        rows.append({
            'layers': n_layers,
            'FNO_params': fno.count_parameters(),
            'FFNO_params': ffno.count_parameters(),
            'ratio': fno.count_parameters() / ffno.count_parameters(),
        })
        del fno, ffno
    df_params = pd.DataFrame(rows)
    df_params.to_csv(RESULTS_DIR / 'parameter_counts.csv', index=False)
    print('Parameter counts:')
    print(df_params.to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(df_params['layers'], df_params['FNO_params'] / 1e6, 'o-', label='FNO', lw=2, markersize=8)
    ax.plot(df_params['layers'], df_params['FFNO_params'] / 1e6, 's-', label='F-FNO', lw=2, markersize=8)
    ax.set_xlabel('Number of layers')
    ax.set_ylabel('Parameters (millions, log scale)')
    ax.set_title('Parameter count vs depth (H=32, M=12)')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(alpha=0.3, which='both')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'param_count.png', dpi=120)
    plt.show()

In [ ]:
if RUN_PLOTS:
    # ---------- Plot 2: depth sweep ----------
    all_results = torch.load(RESULTS_DIR / 'depth_sweep_results.pt', weights_only=False)
    rows = []
    for key, r in all_results.items():
        model_type, layer_tag = key.split('_')
        rows.append({
            'model': model_type,
            'layers': int(layer_tag.lstrip('L')),
            'test_NMSE': r['test_loss'],
            'best_val_NMSE': r['best_val_loss'],
            'params': r['n_params'],
            'time_sec': r['total_time_sec'],
        })
    df = pd.DataFrame(rows).sort_values(['model', 'layers'])
    df.to_csv(RESULTS_DIR / 'depth_sweep_summary.csv', index=False)
    print('Depth sweep results:')
    print(df.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for model_type, color in [('fno', 'C0'), ('ffno', 'C1')]:
        sub = df[df.model == model_type].sort_values('layers')
        label = 'FNO' if model_type == 'fno' else 'F-FNO'
        axes[0].plot(sub.layers, sub.test_NMSE * 100, 'o-', color=color, lw=2, label=label, markersize=8)
    axes[0].set_xlabel('Number of layers')
    axes[0].set_ylabel('Test N-MSE (%)')
    axes[0].set_title('Accuracy vs depth')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    for model_type, color in [('fno', 'C0'), ('ffno', 'C1')]:
        sub = df[df.model == model_type].sort_values('params')
        label = 'FNO' if model_type == 'fno' else 'F-FNO'
        axes[1].plot(sub.params / 1e6, sub.test_NMSE * 100, 'o-', color=color, lw=2, label=label, markersize=8)
        for _, row in sub.iterrows():
            axes[1].annotate(f"L={int(row.layers)}", (row.params / 1e6, row.test_NMSE * 100),
                             textcoords='offset points', xytext=(6, 4), fontsize=9)
    axes[1].set_xlabel('Parameters (millions, log scale)')
    axes[1].set_ylabel('Test N-MSE (%)')
    axes[1].set_title('Parameter efficiency')
    axes[1].set_xscale('log')
    axes[1].legend()
    axes[1].grid(alpha=0.3, which='both')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'depth_comparison.png', dpi=120)
    plt.show()

In [ ]:
if RUN_PLOTS:
    # ---------- Plot 3: training curves ----------
    fig, axes = plt.subplots(1, len(DEPTHS), figsize=(5 * len(DEPTHS), 4), sharey=True)
    if len(DEPTHS) == 1:
        axes = [axes]
    for ax, L in zip(axes, DEPTHS):
        for model_type, color in [('fno', 'C0'), ('ffno', 'C1')]:
            key = f'{model_type}_L{L}'
            if key not in all_results:
                continue
            hist = all_results[key]['history']
            if len(hist['val_loss']) > 0:
                # val_loss has one entry per epoch; align with corresponding steps
                val_steps = hist['step'][-len(hist['val_loss']):]
                ax.plot(val_steps, hist['val_loss'], '-', color=color,
                        label=f"{'F-FNO' if model_type == 'ffno' else 'FNO'}", lw=2)
        ax.set_title(f'{L} layers')
        ax.set_xlabel('Training step')
        ax.set_yscale('log')
        ax.grid(alpha=0.3, which='both')
        ax.legend()
    axes[0].set_ylabel('Validation N-MSE')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=120)
    plt.show()

In [ ]:
if RUN_PLOTS:
    # ---------- Plot 4: ablation bar chart ----------
    ablation_results = torch.load(RESULTS_DIR / 'ablation_results.pt', weights_only=False)
    rows = []
    for name, r in ablation_results.items():
        cfg = r['config']
        rows.append({
            'config': name,
            'model': cfg['model_type'],
            'cosine_LR': cfg['use_cosine_decay'],
            'noise_std': cfg['input_noise_std'],
            'test_NMSE': r['test_loss'],
            'params': r['n_params'],
        })
    df_abl = pd.DataFrame(rows)
    df_abl.to_csv(RESULTS_DIR / 'ablation_summary.csv', index=False)
    print('Ablation results:')
    print(df_abl.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['C0' if m == 'fno' else 'C1' for m in df_abl.model]
    y_pos = np.arange(len(df_abl))
    ax.barh(y_pos, df_abl.test_NMSE * 100, color=colors, alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_abl.config)
    ax.invert_yaxis()
    ax.set_xlabel('Test N-MSE (%)')
    ax.set_title('Training-trick ablation (8-layer models)')
    for i, v in enumerate(df_abl.test_NMSE * 100):
        ax.text(v + 0.2, i, f'{v:.2f}%', va='center', fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='C0', label='FNO'), Patch(color='C1', label='F-FNO')],
              loc='lower right')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'ablation_bars.png', dpi=120)
    plt.show()

In [ ]:
if RUN_PLOTS:
    # ---------- Plot 5: rollout error vs time ----------
    from data.dataset import RolloutDataset
    from train import evaluate_rollout, build_model, TrainConfig

    L_pick = 8
    train_data_tmp = torch.load(DATA_DIR / 'train.pt', weights_only=True)
    stats = {'mean': train_data_tmp.mean().item(), 'std': train_data_tmp.std().item()}
    del train_data_tmp
    rollout_ds = RolloutDataset(DATA_DIR / 'test.pt', normalize=True, stats=stats)

    rollout_results = {}
    for model_type in ['fno', 'ffno']:
        key = f'{model_type}_L{L_pick}'
        if key not in all_results:
            print(f'Skipping rollout for {key} — not trained.')
            continue
        cfg = TrainConfig(**all_results[key]['config'])
        model = build_model(cfg).to(device)
        model.load_state_dict(torch.load(RUNS_DIR / key / 'best.pt', weights_only=True))
        rollout_results[model_type] = evaluate_rollout(model, rollout_ds, device=device)
        del model

    plt.figure(figsize=(8, 4.5))
    for model_type, label, color in [('fno', 'FNO', 'C0'), ('ffno', 'F-FNO', 'C1')]:
        if model_type not in rollout_results:
            continue
        m = np.array(rollout_results[model_type]['per_step_mean']) * 100
        s = np.array(rollout_results[model_type]['per_step_std']) * 100
        steps = np.arange(1, len(m) + 1)
        plt.plot(steps, m, 'o-', color=color, label=label, lw=2, markersize=7)
        plt.fill_between(steps, m - s, m + s, color=color, alpha=0.2)
    plt.xlabel('Rollout step')
    plt.ylabel('N-MSE (%)')
    plt.title(f'Autoregressive rollout error ({L_pick}-layer models)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'rollout_error.png', dpi=120)
    plt.show()

In [ ]:
if RUN_PLOTS:
    # ---------- Plot 6: qualitative rollout side-by-side ----------
    test_data = torch.load(DATA_DIR / 'test.pt', weights_only=True).to(device)
    traj = test_data[0]  # pick trajectory 0
    coords = rollout_ds.coords.to(device)
    mean, std = stats['mean'], stats['std']

    rollout_preds = {}
    for model_type in ['fno', 'ffno']:
        key = f'{model_type}_L{L_pick}'
        if key not in all_results:
            continue
        cfg = TrainConfig(**all_results[key]['config'])
        model = build_model(cfg).to(device)
        model.load_state_dict(torch.load(RUNS_DIR / key / 'best.pt', weights_only=True))
        model.eval()
        preds = [traj[0]]
        w = ((traj[0] - mean) / std).unsqueeze(0)
        with torch.no_grad():
            for t in range(1, traj.size(0)):
                x_in = torch.cat([w, coords.unsqueeze(0)], dim=1)
                w = model(x_in).squeeze(1)
                preds.append(w.squeeze(0) * std + mean)
        rollout_preds[model_type] = torch.stack(preds)
        del model

    if len(rollout_preds) == 2:
        show_t = [1, 5, 10, 15, 19]
        fig, axes = plt.subplots(3, len(show_t), figsize=(3.2 * len(show_t), 9))
        vmax = traj.abs().max().item()
        for col, t in enumerate(show_t):
            axes[0, col].imshow(traj[t].cpu(), cmap='RdBu_r', vmin=-vmax, vmax=vmax)
            axes[0, col].set_title(f't = {t * 0.1:.1f}s')
            axes[1, col].imshow(rollout_preds['fno'][t].cpu(), cmap='RdBu_r', vmin=-vmax, vmax=vmax)
            axes[2, col].imshow(rollout_preds['ffno'][t].cpu(), cmap='RdBu_r', vmin=-vmax, vmax=vmax)
            for row in range(3):
                axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        axes[0, 0].set_ylabel('Ground truth', fontsize=12)
        axes[1, 0].set_ylabel('FNO', fontsize=12)
        axes[2, 0].set_ylabel('F-FNO', fontsize=12)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'qualitative_rollout.png', dpi=120, bbox_inches='tight')
        plt.show()
    del test_data

## Done!

All outputs are in `mini_project/results/`:

- `parameter_counts.csv`
- `depth_sweep_summary.csv` + `depth_sweep_results.pt`
- `ablation_summary.csv` + `ablation_results.pt`
- `sample_trajectory.png`
- `param_count.png`
- `depth_comparison.png`
- `training_curves.png`
- `ablation_bars.png`
- `rollout_error.png`
- `qualitative_rollout.png`

Send these files (especially the CSVs and the main PNG plots) back to
Claude to analyze the results and draft Section 3 + Section 4 of the
report.